In [ ]:
# Ensure repository-root working directory after notebook relocation
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
repo_root = cwd
if not (repo_root / 'pyproject.toml').exists():
    for parent in [cwd, *cwd.parents]:
        if (parent / 'pyproject.toml').exists():
            repo_root = parent
            break
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f'Working directory: {repo_root}')


## Preparing all data for config file

In [2]:
!download_public_data data

uspto_model.onnx: 100%|████████████████████| 91.5M/91.5M [00:17<00:00, 5.22MB/s]
uspto_templates.csv.gz: 3.31MB [00:01, 1.68MB/s]
uspto_ringbreaker_model.onnx: 100%|████████| 15.0M/15.0M [00:11<00:00, 1.33MB/s]
uspto_ringbreaker_templates.csv.gz: 374kB [00:00, 3.24MB/s]
zinc_stock.hdf5: 100%|███████████████████████| 663M/663M [00:57<00:00, 11.5MB/s]
uspto_filter_model.onnx: 100%|█████████████| 16.8M/16.8M [00:04<00:00, 3.51MB/s]
Configuration file written to config.yml


In [4]:
!gzip -dk ./data/uspto_templates.csv.gz

## Converting ASKCOS'S buyable database to AiZynthFinder's and SynPlanner's

In [ ]:
import json

input_file = f'./data/buyables_all.json' # From ASKCOS  https://gitlab.com/mlpds_mit/askcosv2/retro/template_relevance

with open(input_file, 'r') as file:
    molecules = json.load(file)

print('Total molecules in DB in ASKCOS:', len(molecules))

Total molecules in DB in ASKCOS: 329635


In [2]:
# Extract SMILES
buy_db_smiles= [x['smiles'] for x in molecules]

with open("./data/buy_db_askcos.smi", "w", encoding="utf-8") as f:
    for s in buy_db_smiles:
        f.write(f"{s}\n")

### Converting molecules DB for AiZynthFinder

In [3]:
!smiles2stock --files data/buy_db_askcos.smi --output data/buy_db_askcos.hdf5

Processing data/buy_db_askcos.smi
/Users/almazgil/miniforge3-arm/envs/az_synplan_env/lib/python3.11/site-packages/aizynthfinder/tools/make_stock.py:117: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  data.to_hdf(filename, "table")
Created HDF5 stock with 311184 unique compounds


### Converting molecules DB for SynPlanner

In [ ]:
from CGRtools import smiles
from synplan.chem.utils import mol_from_smiles
from tqdm import tqdm 

stand_buy_db_smiles = []
for mol in tqdm(buy_db_smiles):
    try:
        stand_buy_db_smiles.append(str(mol_from_smiles(mol)))
    except:
        pass
print('Successfully processed molecules:', len(stand_buy_db_smiles))

100%|██████████| 329635/329635 [11:55<00:00, 460.52it/s]


329560

In [ ]:
with open("./data/buy_db_askcos_stand.smi", "w", encoding="utf-8") as f:
    for s in stand_buy_db_smiles:
        f.write(f"{s}\n")

### Search if mol in BB

In [ ]:
def mol_in_stocks(target_mol):
    present = False
    for mol in tqdm(stand_buy_db_smiles):
        if str(smiles(target_mol)) == mol:
            print('yes')
            present = True
            break
    if not present:
        print('no')

target_mol = 'N#CC1(c2ccc(NC(=O)c3cccnc3NCc3ccncc3)cc2)CCCC1'
mol_in_stocks(target_mol)  # Example molecule
